In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")

In [ ]:
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

In [ ]:
train_df["text"] = train_df["prompt"] + ' ' + train_df["response_a"] + ' ' + train_df["response_b"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(train_df['text'])

In [ ]:
y  = train_df[["winner_model_a", "winner_model_b", "winner_tie"]]
y = np.argmax(y, axis = 1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier on the training set
clf.fit(X_train, y_train)

In [ ]:
# Predict on the validation set
y_pred = clf.predict(X_val)

# Evaluate performance with classification report and accuracy
print("Classification Report:\n", classification_report(y_val, y_pred))
print("Accuracy Score:", accuracy_score(y_val, y_pred))

In [ ]:
# Combine 'prompt', 'response_a', and 'response_b' for test data
test_df["text"] = test_df["prompt"] + ' ' + test_df["response_a"] + ' ' + test_df["response_b"]

# Transform test data using the same vectorizer
X_test = vectorizer.transform(test_df['text'])

In [ ]:
# Predict class probabilities
probs = clf.predict_proba(X_test)

# Check a few example rows
probs[:5]

In [ ]:
# Create the submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie': probs[:, 2],
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

# Preview
submission.head()